# 🔍 Sistema RAG con Generación SQL Automática

Este notebook implementa un sistema RAG completo que:
- **Genera SQL automáticamente** desde consultas en lenguaje natural
- Usa **bases de datos reales** (PostgreSQL/SQLite intercambiables)
- Implementa **arquitectura de 3 módulos**: Expansión → Recuperación → Generación
- Soporte para **múltiples LLMs**: OpenAI, Gemini, modelos locales
- Integración con **VannaAI** y **PandasAI**

## 🏗️ Arquitectura del Sistema

```
Consulta Usuario → [Query Expansion] → [SQL Generation] → [DB Query] → [Response Generation]
```

### Módulos:
1. **🔍 Query Expansion**: Mejora la consulta del usuario
2. **🗄️ SQL Generation**: Convierte lenguaje natural a SQL
3. **📊 Database Retrieval**: Ejecuta SQL y obtiene datos
4. **🤖 Response Generation**: Genera respuesta en lenguaje natural

## 📦 Instalación de Dependencias

In [1]:
# Instalar todas las dependencias necesarias
!pip install -q sqlalchemy psycopg2-binary pandas
!pip install -q langchain langchain-community langchain-openai
!pip install -q google-generativeai
!pip install -q vanna pandasai
!pip install -q faiss-cpu chromadb
!pip install -q python-dotenv

  DEPRECATION: Building 'flasgger' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'flasgger'. Discussion can be found at https://github.com/pypa/pip/issues/6334


## ⚙️ Configuración Inicial

In [ ]:
import os
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# Configuración de APIs (reemplaza con tus claves)
os.environ["OPENAI_API_KEY"] = "sk-..."  # Tu clave de OpenAI
os.environ["GOOGLE_API_KEY"] = "..."     # Tu clave de Google

# Configuración de base de datos
DATABASE_TYPE = "sqlite"  # Cambiar a "postgresql" si prefieres
DATABASE_URL = "sqlite:///./empresa_datos.db"  # Base local para demo

print("🚀 Configuración completada")

## 🗄️ Crear Base de Datos de Ejemplo

Vamos a crear una base de datos realista con datos de una empresa para probar el sistema.

In [ ]:
# Crear conexión a la base de datos
engine = create_engine(DATABASE_URL)

# Datos de ejemplo - Empleados
empleados_data = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8],
    'nombre': ['Ana García', 'Carlos López', 'María Rodríguez', 'Juan Pérez', 
               'Laura Martín', 'Diego Silva', 'Carmen Torres', 'Roberto Díaz'],
    'edad': [28, 35, 31, 42, 26, 38, 33, 29],
    'departamento': ['IT', 'Ventas', 'IT', 'RRHH', 'Marketing', 'Ventas', 'IT', 'Marketing'],
    'salario': [75000, 65000, 80000, 70000, 60000, 72000, 85000, 58000],
    'ciudad': ['Madrid', 'Barcelona', 'Madrid', 'Valencia', 'Sevilla', 'Barcelona', 'Madrid', 'Bilbao'],
    'fecha_ingreso': ['2020-01-15', '2019-03-20', '2021-06-10', '2018-11-05', 
                     '2022-02-28', '2020-08-12', '2019-09-30', '2021-12-03']
}

# Datos de ejemplo - Proyectos
proyectos_data = {
    'id': [1, 2, 3, 4, 5],
    'nombre': ['Sistema CRM', 'App Móvil', 'Dashboard Analytics', 'Portal Web', 'API Gateway'],
    'presupuesto': [150000, 80000, 120000, 200000, 95000],
    'estado': ['Completado', 'En Progreso', 'Completado', 'En Progreso', 'Planificado'],
    'responsable_id': [3, 1, 7, 3, 1],
    'fecha_inicio': ['2023-01-01', '2023-06-15', '2023-03-10', '2023-09-01', '2024-01-15']
}

# Datos de ejemplo - Ventas
ventas_data = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8],
    'vendedor_id': [2, 6, 2, 6, 2, 6, 2, 6],
    'cliente': ['Empresa A', 'Empresa B', 'Empresa C', 'Empresa D', 
                'Empresa E', 'Empresa F', 'Empresa G', 'Empresa H'],
    'monto': [25000, 45000, 30000, 35000, 20000, 55000, 40000, 28000],
    'fecha': ['2023-01-15', '2023-02-20', '2023-03-10', '2023-04-05',
              '2023-05-12', '2023-06-18', '2023-07-22', '2023-08-30'],
    'producto': ['CRM', 'Dashboard', 'CRM', 'API', 'Dashboard', 'CRM', 'API', 'Dashboard']
}

# Crear DataFrames y guardar en la base de datos
df_empleados = pd.DataFrame(empleados_data)
df_proyectos = pd.DataFrame(proyectos_data)
df_ventas = pd.DataFrame(ventas_data)

# Guardar en la base de datos
df_empleados.to_sql('empleados', engine, if_exists='replace', index=False)
df_proyectos.to_sql('proyectos', engine, if_exists='replace', index=False)
df_ventas.to_sql('ventas', engine, if_exists='replace', index=False)

print("✅ Base de datos creada con éxito")
print(f"📊 Tablas creadas: empleados ({len(df_empleados)} filas), proyectos ({len(df_proyectos)} filas), ventas ({len(df_ventas)} filas)")

# Mostrar esquema de las tablas
with engine.connect() as conn:
    for tabla in ['empleados', 'proyectos', 'ventas']:
        result = conn.execute(text(f"SELECT * FROM {tabla} LIMIT 2"))
        print(f"\n📋 Muestra de tabla '{tabla}':")
        for row in result:
            print(f"  {dict(row)}")

## 🧠 Sistema RAG con Generación SQL

### Módulo 1: Query Expansion

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage
import google.generativeai as genai

class QueryExpansionModule:
    """Módulo 1: Expansión de consultas para mejorar la generación SQL"""
    
    def __init__(self, llm_provider="openai"):
        self.llm_provider = llm_provider
        
        if llm_provider == "openai":
            self.llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)
        elif llm_provider == "gemini":
            genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
            self.model = genai.GenerativeModel("gemini-1.5-flash")
    
    def expand_query(self, user_query: str, schema_info: str) -> str:
        """Expande la consulta del usuario para mejorar la generación SQL"""
        
        system_prompt = f"""
Eres un experto en análisis de consultas SQL. Tu tarea es expandir y mejorar consultas en lenguaje natural para que sean más precisas y completas.

ESQUEMA DE LA BASE DE DATOS:
{schema_info}

INSTRUCCIONES:
1. Analiza la consulta original del usuario
2. Identifica conceptos clave y términos relacionados
3. Sugiere campos y tablas relevantes que podrían ser útiles
4. Expande la consulta manteniendo la intención original
5. Incluye contexto adicional que mejore la precisión del SQL

Devuelve SOLO la consulta expandida, sin explicaciones.
"""
        
        user_prompt = f"Consulta original: {user_query}"
        
        if self.llm_provider == "openai":
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt)
            ]
            response = self.llm(messages)
            return response.content.strip()
        
        elif self.llm_provider == "gemini":
            full_prompt = f"{system_prompt}\n\n{user_prompt}"
            response = self.model.generate_content(full_prompt)
            return response.text.strip()
        
        return user_query  # Fallback

# Test del módulo
schema_info = """
TABLAS:
- empleados: id, nombre, edad, departamento, salario, ciudad, fecha_ingreso
- proyectos: id, nombre, presupuesto, estado, responsable_id, fecha_inicio
- ventas: id, vendedor_id, cliente, monto, fecha, producto
"""

query_expander = QueryExpansionModule(llm_provider="openai")
ejemplo_consulta = "¿Cuánto ganan los empleados de IT?"
consulta_expandida = query_expander.expand_query(ejemplo_consulta, schema_info)

print(f"🔍 Consulta original: {ejemplo_consulta}")
print(f"🔎 Consulta expandida: {consulta_expandida}")

### Módulo 2: SQL Generation

In [ ]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain.chains.sql_database.prompt import PROMPT_SUFFIX, _sqlite_prompt

class SQLGenerationModule:
    """Módulo 2: Generación automática de SQL desde lenguaje natural"""
    
    def __init__(self, database_url: str, llm_provider="openai"):
        self.db = SQLDatabase.from_uri(database_url)
        self.llm_provider = llm_provider
        
        if llm_provider == "openai":
            self.llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        elif llm_provider == "gemini":
            genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
            self.model = genai.GenerativeModel("gemini-1.5-flash")
    
    def generate_sql(self, natural_query: str) -> str:
        """Genera SQL a partir de una consulta en lenguaje natural"""
        
        # Obtener información del esquema
        table_info = self.db.get_table_info()
        dialect = self.db.dialect
        
        system_prompt = f"""
Eres un experto en SQL que convierte consultas en lenguaje natural a SQL válido.

INFORMACIÓN DE LA BASE DE DATOS:
Dialecto: {dialect}
Esquema de tablas:
{table_info}

REGLAS IMPORTANTES:
1. Genera SOLO la consulta SQL, sin explicaciones
2. Usa únicamente las tablas y columnas que existen
3. Aplica LIMIT 10 por defecto a menos que se especifique otro número
4. Usa alias claros para las columnas
5. Para consultas de agregación, incluye GROUP BY cuando sea necesario
6. Ordena los resultados de manera lógica
7. NO incluyas markdown ni formato adicional

EJEMPLOS:
- "empleados de IT" → SELECT * FROM empleados WHERE departamento = 'IT' LIMIT 10;
- "salario promedio por departamento" → SELECT departamento, AVG(salario) as salario_promedio FROM empleados GROUP BY departamento;
"""
        
        user_prompt = f"Consulta: {natural_query}"
        
        if self.llm_provider == "openai":
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt)
            ]
            response = self.llm(messages)
            sql_query = response.content.strip()
        
        elif self.llm_provider == "gemini":
            full_prompt = f"{system_prompt}\n\n{user_prompt}"
            response = self.model.generate_content(full_prompt)
            sql_query = response.text.strip()
        
        # Limpiar la respuesta (remover markdown si existe)
        sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
        if not sql_query.endswith(';'):
            sql_query += ';'
            
        return sql_query
    
    def validate_and_execute(self, sql_query: str) -> tuple:
        """Valida y ejecuta la consulta SQL"""
        try:
            # Ejecutar la consulta
            result = self.db.run(sql_query)
            return True, result, None
        except Exception as e:
            return False, None, str(e)

# Test del módulo SQL
sql_generator = SQLGenerationModule(DATABASE_URL, llm_provider="openai")

# Pruebas con diferentes consultas
test_queries = [
    "¿Cuántos empleados trabajan en IT?",
    "Muéstrame el salario promedio por departamento",
    "¿Cuáles son los proyectos en progreso?"
]

for query in test_queries:
    print(f"\n🔍 Consulta: {query}")
    sql = sql_generator.generate_sql(query)
    print(f"🗄️ SQL generado: {sql}")
    
    success, result, error = sql_generator.validate_and_execute(sql)
    if success:
        print(f"✅ Resultado: {result}")
    else:
        print(f"❌ Error: {error}")

### Módulo 3: Response Generation

In [ ]:
class ResponseGenerationModule:
    """Módulo 3: Generación de respuestas en lenguaje natural"""
    
    def __init__(self, llm_provider="openai"):
        self.llm_provider = llm_provider
        
        if llm_provider == "openai":
            self.llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)
        elif llm_provider == "gemini":
            genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
            self.model = genai.GenerativeModel("gemini-1.5-flash")
    
    def generate_response(self, 
                         original_query: str, 
                         expanded_query: str,
                         sql_query: str, 
                         sql_result: str) -> str:
        """Genera una respuesta en lenguaje natural basada en los resultados SQL"""
        
        system_prompt = """
Eres un analista de datos experto que interpreta resultados de consultas SQL y los presenta de manera clara y comprensible.

INSTRUCCIONES:
1. Responde directamente a la pregunta original del usuario
2. Usa los datos del resultado SQL para dar una respuesta precisa
3. Incluye números específicos y detalles relevantes
4. Si hay múltiples resultados, resúmelos de manera organizada
5. Usa un tono profesional pero accesible
6. Si los datos están vacíos, explica que no se encontraron resultados
7. Incluye insights adicionales si son relevantes
"""
        
        user_prompt = f"""
PREGUNTA ORIGINAL: {original_query}
CONSULTA EXPANDIDA: {expanded_query}
CONSULTA SQL EJECUTADA: {sql_query}
RESULTADOS: {sql_result}

Por favor, proporciona una respuesta clara y completa basada en estos datos.
"""
        
        if self.llm_provider == "openai":
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt)
            ]
            response = self.llm(messages)
            return response.content.strip()
        
        elif self.llm_provider == "gemini":
            full_prompt = f"{system_prompt}\n\n{user_prompt}"
            response = self.model.generate_content(full_prompt)
            return response.text.strip()
        
        return "Error generando respuesta"

# Test del módulo de respuesta
response_generator = ResponseGenerationModule(llm_provider="openai")

# Ejemplo de uso
test_response = response_generator.generate_response(
    original_query="¿Cuántos empleados trabajan en IT?",
    expanded_query="Número total de empleados que trabajan en el departamento de Tecnologías de la Información (IT)",
    sql_query="SELECT COUNT(*) FROM empleados WHERE departamento = 'IT';",
    sql_result="[(3,)]"
)

print(f"🤖 Respuesta generada: {test_response}")

## 🚀 Sistema RAG-SQL Completo

Ahora integramos todos los módulos en un sistema completo.

In [ ]:
import time
from datetime import datetime

class RAGSQLSystem:
    """Sistema RAG completo con generación automática de SQL"""
    
    def __init__(self, database_url: str, llm_provider="openai"):
        self.database_url = database_url
        self.llm_provider = llm_provider
        
        # Inicializar módulos
        self.query_expander = QueryExpansionModule(llm_provider)
        self.sql_generator = SQLGenerationModule(database_url, llm_provider)
        self.response_generator = ResponseGenerationModule(llm_provider)
        
        # Schema info para expansión
        self.schema_info = self._get_schema_info()
        
        print(f"✅ Sistema RAG-SQL inicializado con {llm_provider.upper()}")
    
    def _get_schema_info(self) -> str:
        """Obtiene información del esquema de la base de datos"""
        try:
            return self.sql_generator.db.get_table_info()
        except:
            return "Esquema no disponible"
    
    def process_query(self, user_query: str) -> dict:
        """Procesa una consulta completa a través del pipeline RAG-SQL"""
        start_time = time.time()
        result = {
            'timestamp': datetime.now().isoformat(),
            'original_query': user_query,
            'expanded_query': '',
            'generated_sql': '',
            'sql_result': '',
            'final_response': '',
            'success': False,
            'error': None,
            'timing': {}
        }
        
        try:
            # Paso 1: Expansión de consulta
            step1_start = time.time()
            expanded_query = self.query_expander.expand_query(user_query, self.schema_info)
            result['expanded_query'] = expanded_query
            result['timing']['expansion'] = time.time() - step1_start
            
            # Paso 2: Generación SQL
            step2_start = time.time()
            sql_query = self.sql_generator.generate_sql(expanded_query)
            result['generated_sql'] = sql_query
            result['timing']['sql_generation'] = time.time() - step2_start
            
            # Paso 3: Ejecución SQL
            step3_start = time.time()
            sql_success, sql_result, sql_error = self.sql_generator.validate_and_execute(sql_query)
            
            if not sql_success:
                result['error'] = f"Error SQL: {sql_error}"
                result['final_response'] = f"Lo siento, hubo un error al procesar tu consulta: {sql_error}"
                return result
            
            result['sql_result'] = sql_result
            result['timing']['sql_execution'] = time.time() - step3_start
            
            # Paso 4: Generación de respuesta
            step4_start = time.time()
            final_response = self.response_generator.generate_response(
                user_query, expanded_query, sql_query, sql_result
            )
            result['final_response'] = final_response
            result['timing']['response_generation'] = time.time() - step4_start
            
            result['success'] = True
            
        except Exception as e:
            result['error'] = str(e)
            result['final_response'] = f"Error procesando la consulta: {str(e)}"
        
        result['timing']['total'] = time.time() - start_time
        return result
    
    def chat_interface(self):
        """Interfaz de chat interactiva"""
        print("\n🔍 Sistema RAG-SQL - Chat Interactivo")
        print("Escribe 'salir' para terminar\n")
        
        while True:
            try:
                query = input("👤 Tu pregunta: ").strip()
                
                if query.lower() in ['salir', 'exit', 'quit']:
                    print("👋 ¡Hasta luego!")
                    break
                
                if not query:
                    continue
                
                print("\n⏳ Procesando...")
                result = self.process_query(query)
                
                if result['success']:
                    print(f"\n🤖 Respuesta: {result['final_response']}")
                    print(f"\n📊 SQL ejecutado: {result['generated_sql']}")
                    print(f"⏱️ Tiempo total: {result['timing']['total']:.2f}s")
                else:
                    print(f"\n❌ Error: {result['error']}")
                
                print("\n" + "-"*60)
                
            except KeyboardInterrupt:
                print("\n👋 ¡Hasta luego!")
                break
            except Exception as e:
                print(f"\n❌ Error inesperado: {e}")

# Inicializar el sistema
rag_sql_system = RAGSQLSystem(DATABASE_URL, llm_provider="openai")
print("🎉 Sistema listo para usar")

## 🧪 Pruebas del Sistema

Vamos a probar el sistema con diferentes tipos de consultas.

In [ ]:
# Consultas de prueba
test_queries = [
    "¿Cuántos empleados tenemos en total?",
    "¿Cuál es el salario promedio por departamento?",
    "¿Qué empleados ganan más de 70000?",
    "¿Cuáles son los proyectos en progreso y quién los lidera?",
    "¿Cuánto ha vendido cada vendedor este año?",
    "¿Cuáles son los 3 empleados más jóvenes?",
    "¿Qué departamento tiene el mayor presupuesto en proyectos?"
]

print("🧪 Ejecutando pruebas del sistema RAG-SQL\n")
print("="*80)

for i, query in enumerate(test_queries, 1):
    print(f"\n🔍 PRUEBA {i}: {query}")
    print("-" * 60)
    
    result = rag_sql_system.process_query(query)
    
    if result['success']:
        print(f"✅ ÉXITO")
        print(f"📈 Consulta expandida: {result['expanded_query']}")
        print(f"🗄️ SQL generado: {result['generated_sql']}")
        print(f"📊 Datos obtenidos: {result['sql_result']}")
        print(f"🤖 Respuesta: {result['final_response']}")
        print(f"⏱️ Tiempos - Total: {result['timing']['total']:.2f}s | SQL: {result['timing']['sql_generation']:.2f}s | Respuesta: {result['timing']['response_generation']:.2f}s")
    else:
        print(f"❌ ERROR: {result['error']}")
        if result['generated_sql']:
            print(f"🗄️ SQL intentado: {result['generated_sql']}")
    
    print("\n" + "="*80)

print("\n🎉 Pruebas completadas")

## 🔧 Integración con VannaAI

VannaAI es una librería especializada en generar SQL desde lenguaje natural.

In [ ]:
# Ejemplo con VannaAI (comentado porque requiere configuración adicional)
"""
import vanna
from vanna.openai.openai_chat import OpenAI_Chat
from vanna.chromadb.chromadb_vector import ChromaDB_VectorStore

class VannaRAGSystem:
    def __init__(self, database_url: str):
        # Configurar Vanna con OpenAI y ChromaDB
        class MyVanna(ChromaDB_VectorStore, OpenAI_Chat):
            def __init__(self, config=None):
                ChromaDB_VectorStore.__init__(self, config=config)
                OpenAI_Chat.__init__(self, config=config)
        
        self.vn = MyVanna(config={'api_key': os.environ['OPENAI_API_KEY'], 'model': 'gpt-3.5-turbo'})
        
        # Conectar a la base de datos
        if database_url.startswith('sqlite'):
            self.vn.connect_to_sqlite(database_url.replace('sqlite:///', ''))
        else:
            self.vn.connect_to_postgres(database_url)
    
    def setup_training(self):
        # Entrenar Vanna con el esquema de la base
        df_information_schema = self.vn.run_sql("SELECT * FROM information_schema.tables")
        plan = self.vn.get_training_plan_generic(df_information_schema)
        self.vn.train(plan=plan)
    
    def ask_question(self, question: str):
        # Generar SQL y obtener respuesta
        sql = self.vn.generate_sql(question)
        df = self.vn.run_sql(sql)
        response = self.vn.generate_followup_questions(question, sql, df)
        return sql, df, response

# Descomentar para usar VannaAI:
# vanna_system = VannaRAGSystem(DATABASE_URL)
# vanna_system.setup_training()
# sql, df, response = vanna_system.ask_question("¿Cuántos empleados hay por departamento?")
# print(f"SQL: {sql}")
# print(f"Datos: {df}")
"""

print("📝 Código de integración con VannaAI preparado (comentado)")
print("   Para usar VannaAI, descomenta el código y configura las credenciales")

## 🐼 Integración con PandasAI

In [ ]:
# Ejemplo con PandasAI
"""
from pandasai import SmartDataframe
from pandasai.llm import OpenAI

class PandasAIRAGSystem:
    def __init__(self, api_key: str):
        self.llm = OpenAI(api_token=api_key)
    
    def analyze_dataframe(self, df: pd.DataFrame, question: str):
        smart_df = SmartDataframe(df, config={"llm": self.llm})
        response = smart_df.chat(question)
        return response

# Ejemplo de uso con nuestros datos
# pandas_ai_system = PandasAIRAGSystem(os.environ['OPENAI_API_KEY'])
# 
# # Cargar datos desde la base
# df_empleados = pd.read_sql("SELECT * FROM empleados", engine)
# 
# # Hacer pregunta
# response = pandas_ai_system.analyze_dataframe(
#     df_empleados, 
#     "¿Cuál es la distribución de salarios por departamento?"
# )
# print(response)
"""

print("📊 Código de integración con PandasAI preparado (comentado)")
print("   PandasAI es excelente para análisis exploratorio de DataFrames")

## 🚀 Chat Interactivo

¡Probemos el sistema con un chat interactivo!

In [ ]:
# Versión simplificada para notebooks
def demo_chat():
    """Demo del chat para el notebook"""
    demo_questions = [
        "¿Cuántos empleados tenemos?",
        "¿Quién es el empleado mejor pagado?",
        "¿Cuáles son las ventas totales por producto?",
        "¿Qué proyectos están en progreso?"
    ]
    
    print("🎮 DEMO - Chat RAG-SQL\n")
    
    for i, question in enumerate(demo_questions, 1):
        print(f"👤 Usuario: {question}")
        
        result = rag_sql_system.process_query(question)
        
        if result['success']:
            print(f"🤖 Asistente: {result['final_response']}")
            print(f"📄 SQL: {result['generated_sql']}")
        else:
            print(f"❌ Error: {result['error']}")
        
        print("\n" + "-"*50 + "\n")

# Ejecutar demo
demo_chat()

print("\n💡 Para usar el chat interactivo completo, ejecuta:")
print("   rag_sql_system.chat_interface()")

## 📊 Análisis de Rendimiento

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Analizar tiempos de respuesta
def analyze_performance():
    """Analiza el rendimiento del sistema"""
    test_queries = [
        "¿Cuántos empleados hay?",
        "Salario promedio por departamento",
        "Ventas totales por vendedor",
        "Proyectos completados",
        "Empleados de Madrid"
    ]
    
    results = []
    
    for query in test_queries:
        result = rag_sql_system.process_query(query)
        if result['success']:
            results.append({
                'query': query[:20] + '...',
                'total_time': result['timing']['total'],
                'expansion_time': result['timing']['expansion'],
                'sql_gen_time': result['timing']['sql_generation'],
                'sql_exec_time': result['timing']['sql_execution'],
                'response_time': result['timing']['response_generation']
            })
    
    # Crear DataFrame para análisis
    df_performance = pd.DataFrame(results)
    
    print("📊 Análisis de Rendimiento:")
    print(df_performance.round(3))
    
    # Estadísticas
    print(f"\n📈 Estadísticas:")
    print(f"   Tiempo promedio total: {df_performance['total_time'].mean():.3f}s")
    print(f"   Tiempo más rápido: {df_performance['total_time'].min():.3f}s")
    print(f"   Tiempo más lento: {df_performance['total_time'].max():.3f}s")
    
    return df_performance

# Ejecutar análisis
performance_data = analyze_performance()

## 🔧 Configuración para Producción

In [ ]:
# Configuración para PostgreSQL en producción
def setup_postgresql():
    """
    Configuración para PostgreSQL en producción
    """
    config_example = """
    # Configuración PostgreSQL
    POSTGRES_CONFIG = {
        'host': 'localhost',
        'port': 5432,
        'database': 'empresa_db',
        'username': 'tu_usuario',
        'password': 'tu_password'
    }
    
    # URL de conexión
    postgres_url = f"postgresql://{POSTGRES_CONFIG['username']}:{POSTGRES_CONFIG['password']}@{POSTGRES_CONFIG['host']}:{POSTGRES_CONFIG['port']}/{POSTGRES_CONFIG['database']}"
    
    # Inicializar sistema con PostgreSQL
    rag_system_prod = RAGSQLSystem(postgres_url, llm_provider="openai")
    """
    
    print("🐘 Configuración PostgreSQL:")
    print(config_example)

def setup_gemini():
    """
    Configuración para usar Gemini en lugar de OpenAI
    """
    config_example = """
    # Configurar Gemini
    import google.generativeai as genai
    
    genai.configure(api_key="tu_google_api_key")
    
    # Inicializar con Gemini
    rag_system_gemini = RAGSQLSystem(DATABASE_URL, llm_provider="gemini")
    
    # Para embeddings con Gecko
    def get_embeddings(texts):
        embeddings = []
        for text in texts:
            response = genai.embed_content(
                model="models/embedding-001",
                content=text,
                task_type="retrieval_document"
            )
            embeddings.append(response['embedding'])
        return embeddings
    """
    
    print("🤖 Configuración Gemini:")
    print(config_example)

# Mostrar configuraciones
setup_postgresql()
print("\n" + "="*60 + "\n")
setup_gemini()

## 🎯 Conclusiones y Próximos Pasos

### ✅ Lo que hemos implementado:

1. **🔍 Sistema RAG completo** con generación automática de SQL
2. **🏗️ Arquitectura de 3 módulos**: Query Expansion → SQL Generation → Response Generation
3. **🗄️ Soporte multi-DB**: SQLite y PostgreSQL intercambiables
4. **🤖 Multi-LLM**: OpenAI y Gemini configurables
5. **📊 Análisis de rendimiento** y métricas detalladas
6. **🧪 Testing comprehensivo** con múltiples tipos de consultas

### 🚀 Mejoras posibles:

1. **📈 Cache de consultas** para mejor rendimiento
2. **🔐 Validación de seguridad** SQL injection prevention
3. **📊 Visualización automática** de resultados
4. **🎯 Fine-tuning** de prompts por dominio
5. **🔄 Feedback loop** para mejorar generación SQL
6. **📱 API REST** para integración externa

### 💡 Casos de uso:

- **📊 Business Intelligence**: Consultas ad-hoc en lenguaje natural
- **📈 Análisis de datos**: Exploración rápida de datasets
- **🎯 Dashboards inteligentes**: Interfaces conversacionales
- **📋 Reportes automáticos**: Generación de informes por voz/texto

---

**🎉 ¡Sistema RAG-SQL implementado con éxito!**

Este notebook demuestra un **sistema RAG completo** que resuelve el problema original: **generar SQL automáticamente** desde consultas en lenguaje natural usando una **arquitectura de 3 módulos** robusta y escalable.